# A3E-DNS v2 walkthrough
Run the unit tests, one experiment and inspect the output. All results are synthetic simulation outputs.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('../src')); sys.path.insert(0, os.path.abspath('../experiments_v2'))
import numpy as np
from a3e_dns_v2 import traffic as T
from a3e_dns_v2.aee import AEEEngine
from a3e_dns_v2.qhe import PARAM_SETS, RLWEAdditive

## 1. Adaptive key evolution

In [ ]:
e = AEEEngine(seed=1)
f = np.random.default_rng(0).standard_normal(16)
for i in range(5):
    k, iv, cad, rec = e.update(f, [5,4,3,2], [4,4,3,3])
    print(k.hex()[:16], round(cad,2), round(rec['sigma'],3))

## 2. Homomorphic addition and noise

In [ ]:
ps = PARAM_SETS['ntt_n256']; sc = RLWEAdditive(ps, 1)
m = np.zeros(ps.n, dtype=np.int64); m[:3] = [4,5,6]
c = sc.encrypt(m)
for _ in range(20): c = sc.add(c, sc.encrypt(m))
print(sc.decrypt(c)[:3], 'expected', (21*m[:3]) % ps.t, 'noise', sc.noise(c, (21*m) % ps.t))

## 3. Synthetic scenarios

In [ ]:
X, y, p, s = T.make_dataset(100, np.random.default_rng(0))
print(X.shape, y.mean(), list(T.SCENARIOS)[:4])

## 4. Run an experiment
The full suite takes a few minutes: `make -f Makefile.v2 results`.

In [ ]:
import e5_aee
e5_aee.key_quality()